# 🛡️ AGL Security Tool — Full Power Cloud Audit
# أداة AGL للتدقيق الأمني — فحص سحابي بالقوة الكاملة

This notebook runs your **agl_security_tool** at full power on Google Colab.

**12 Analysis Layers** | **22 Internal Detectors** | **4 External Tools**

| Layer | Engine | What it does |
|-------|--------|-------------|
| 0 | Import Flattener | Resolves imports + inheritance |
| 0.5 | Z3 Symbolic | SMT formal verification |
| 1 | Smart Contract Analyzer | Lexer + CFG + Pattern |
| 2 | Security Orchestrator | Slither + Mythril + Semgrep |
| 3 | Offensive Security | EVM Sim + Formal Proofs |
| 4 | 22 Detectors | Reentrancy, Access, DeFi, Token, Common |
| L1 | State Extraction | Financial state graph |
| L2 | Action Space | Attack surface enumeration |
| L3 | Attack Simulation | Economic physics / profit calc |
| L4 | Search Engine | Beam/MCTS/Evolutionary search |
| 5 | Exploit Reasoning | Attack chain + Z3 proof |
| 5+ | LLM Enrichment | Explanations + fixes + PoC |

---

## 📦 Step 1: Install Everything / تثبيت كل شيء

In [ ]:
%%time
# Install all dependencies
!pip install -q z3-solver requests slither-analyzer semgrep solc-select pytest

# Install Solidity compiler
!solc-select install 0.8.19 && solc-select use 0.8.19

# Optional: Mythril (heavy but powerful EVM symbolic execution)
# Uncomment the next line if you want Mythril:
# !pip install -q mythril

print("\n✅ Dependencies installed / تم تثبيت الاعتمادات")

## 🔧 Step 2: Clone AGL Security Tool / استنساخ الأداة

In [ ]:
# ═══════════════════════════════════════════════════════
# 👇 CHANGE THIS to your repo URL
# 👇 غيّر هذا إلى رابط مستودعك
# ═══════════════════════════════════════════════════════
TOOL_REPO = "https://github.com/alanqamarqo-dev/your-repo.git"
BRANCH = "agl_security"  # or "master"

import os
if not os.path.exists("/content/agl-tool"):
    !git clone --branch {BRANCH} --depth 1 {TOOL_REPO} /content/agl-tool
else:
    print("Already cloned ✅")

os.chdir("/content/agl-tool")
!ls agl_security_tool/
print(f"\n✅ Tool ready at {os.getcwd()}")

In [3]:
# Verify everything works
import sys
sys.path.insert(0, '/content/agl-tool')

from agl_security_tool import AGLSecurityAudit, __version__
from agl_security_tool.detectors import DetectorRunner

audit = AGLSecurityAudit()
dr = DetectorRunner()

print(f"🛡️ AGL Security Tool v{__version__}")
print(f"🔍 {len(dr.detectors)} detectors loaded")
print()

# Verify external tools
import subprocess
for tool, cmd in [("solc", "solc --version"), ("slither", "slither --version"), ("semgrep", "semgrep --version")]:
    try:
        r = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=10)
        v = r.stdout.strip().split('\n')[0][:50]
        print(f"  ✅ {tool}: {v}")
    except: print(f"  ⚠️ {tool}: not found")

print("\n✅ All systems ready / كل الأنظمة جاهزة")

⚠️ [OffensiveSecurity]: HolographicLLM not found. Reverting to manual mode.
⚠️ [OffensiveSecurity]: AdvancedMetaReasonerEngine not found. Strategy disabled.
⚠️ [OffensiveSecurity]: ResonanceOptimizer not found. Resonance scanning disabled.
⚠️ [OffensiveSecurity]: QuantumNeuralCore not found. Deep Logic Analysis disabled.
   -> 🛡️ AGL Security Suite: LOADED (Slither/Mythril/Semgrep/Z3 Ready)
   -> ⚠️ Strict Logic not available: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.
🔬 [FormalVerifier] Initialized with Z3 SMT Solver
   Timeout: 10000ms
   Invariants: 8 predefined
   -> 🛡️ AGL Security Suite: INITIALIZED
⚔️ [OffensiveSecurityEngine]: Loaded. Neuro-Cybernetic Interface Online.
   -> 🔬 Z3 Formal Verifier: ACTIVE (Mathematical Proofs)
   -> 🛡️ Security Suite: ACTIVE (Slither/Mythril/Semgrep/AST)
🛡️ AGL Security Tool v1.1.0
🔍 22 detectors loaded

  

## 🎯 Step 3: Choose Your Target / اختر الهدف

### Option A: Git URL / رابط مستودع
```
TARGET = "https://github.com/owner/defi-protocol"
```

### Option B: Upload .sol file / ارفع ملف
Use the file upload panel on the left →

### Option C: Practice targets / أهداف للتمرين
```
TARGET = "https://github.com/tinchoabbate/damn-vulnerable-defi"
```

### 🏆 Bug Bounty Platforms / منصات المكافآت
| Platform | URL |
|----------|-----|
| Immunefi | https://immunefi.com/explore/ |
| Code4rena | https://code4rena.com/contests |
| Sherlock | https://audits.sherlock.xyz/contests |
| Hats Finance | https://hats.finance/ |
| Cantina | https://cantina.xyz/ |

In [ ]:
# ═══════════════════════════════════════════════════════
# 👇 SET YOUR TARGET HERE / حدد هدفك هنا
# ═══════════════════════════════════════════════════════

# Option 1: Git repo URL
TARGET = "https://github.com/tinchoabbate/damn-vulnerable-defi"  # Practice target

# Option 2: Local file (after uploading to Colab)
# TARGET = "/content/uploaded_contract.sol"

# Option 3: Specific bounty contest
# TARGET = "https://github.com/code-423n4/2024-01-some-contest"

# ═══════════════════════════════════════════════════════
# Scan mode: "deep" (all 12 layers) | "scan" (standard) | "quick" (fast patterns)
MODE = "deep"
# ═══════════════════════════════════════════════════════

print(f"🎯 Target: {TARGET}")
print(f"🔬 Mode: {MODE}")

In [ ]:
# Clone the target
import os

if TARGET.startswith("http") or TARGET.startswith("git@"):
    repo_name = TARGET.rstrip('/').split('/')[-1].replace('.git', '')
    target_dir = f"/content/targets/{repo_name}"
    if not os.path.exists(target_dir):
        !git clone --depth 1 {TARGET} {target_dir}
    TARGET_PATH = target_dir
elif os.path.exists(TARGET):
    TARGET_PATH = TARGET
else:
    print(f"❌ Target not found: {TARGET}")
    TARGET_PATH = None

if TARGET_PATH:
    # Count .sol files
    sol_count = 0
    for root, dirs, files in os.walk(TARGET_PATH):
        dirs[:] = [d for d in dirs if d not in {'node_modules', '.git', 'lib', 'forge-std'}]
        sol_count += sum(1 for f in files if f.endswith('.sol'))
    print(f"\n📊 Found {sol_count} Solidity files in {TARGET_PATH}")

## 🔍 Step 4: Run Full Audit / تشغيل الفحص الكامل

This will run all 12 analysis layers on every Solidity file.

In [ ]:
%%time
os.chdir("/content/agl-tool")

# Run the automated audit script
!python scripts/colab_full_audit.py \
    --target "{TARGET_PATH}" \
    --mode {MODE} \
    --output /content/audit_reports \
    --skip-install

## 📊 Step 5: View Results / عرض النتائج

In [ ]:
import glob, json
from IPython.display import Markdown, display

# Find latest report
reports = sorted(glob.glob("/content/audit_reports/audit_*.md"))
if reports:
    latest = reports[-1]
    print(f"📝 Latest report: {latest}")
    with open(latest) as f:
        display(Markdown(f.read()))
else:
    print("❌ No reports found")

In [ ]:
# View JSON results
json_reports = sorted(glob.glob("/content/audit_reports/audit_*.json"))
if json_reports:
    with open(json_reports[-1]) as f:
        data = json.load(f)

    # Summary
    total = sum(1 for r in data if 'result' in r
               for _ in r['result'].get('all_findings_unified', r['result'].get('findings', [])))
    critical = sum(1 for r in data if 'result' in r
                   for f in r['result'].get('all_findings_unified', r['result'].get('findings', []))
                   if f.get('severity','').upper() == 'CRITICAL')
    high = sum(1 for r in data if 'result' in r
              for f in r['result'].get('all_findings_unified', r['result'].get('findings', []))
              if f.get('severity','').upper() == 'HIGH')

    print(f"📊 Total findings: {total}")
    print(f"🔴 CRITICAL: {critical}")
    print(f"🟠 HIGH: {high}")
    print(f"\n🎯 Bounty-worthy findings: {critical + high}")

## 💾 Step 6: Download Report / تحميل التقرير

In [ ]:
from google.colab import files
import glob

# Download all reports
for f in glob.glob("/content/audit_reports/*"):
    print(f"📥 Downloading: {os.path.basename(f)}")
    files.download(f)

print("\n✅ All reports downloaded / تم تحميل جميع التقارير")

## 🔄 Run on Another Target / فحص هدف آخر

Change `TARGET` in Step 3 and re-run cells 4-6.

In [ ]:
# Quick scan a single file (paste Solidity code here)
CONTRACT_CODE = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.19;

// Paste your contract here...
contract Example {
    mapping(address => uint256) public balances;

    function withdraw() external {
        uint256 bal = balances[msg.sender];
        (bool success, ) = msg.sender.call{value: bal}("");
        require(success);
        balances[msg.sender] = 0;
    }
}
"""

# Write and scan
with open("/tmp/quick_scan.sol", "w") as f:
    f.write(CONTRACT_CODE)

os.chdir("/content/agl-tool")
from agl_security_tool import AGLSecurityAudit
audit = AGLSecurityAudit({"skip_llm": True})
result = audit.deep_scan("/tmp/quick_scan.sol")

print(f"\n🔍 Findings: {result.get('total_findings', 0)}")
for f in result.get('all_findings_unified', result.get('findings', [])):
    sev = f.get('severity', '?').upper()
    title = f.get('title', f.get('text', ''))[:80]
    print(f"  {'🔴' if sev=='CRITICAL' else '🟠' if sev=='HIGH' else '🟡'} [{sev}] {title}")